# Ejemplo 1 de Wenu: primera carta celeste

Este cuaderno presenta el flujo de trabajo básico para crear una carta astronómica
con Wenu.

La carta celeste final incluirá:

- estrellas del catálogo Hipparcos;
- líneas y etiquetas de constelaciones occidentales;
- límites de constelación de la IAU;
- la eclíptica;
- el plano galáctico;
- puntos de referencia celestes;
- una cuadrícula de coordenadas ecuatoriales.

El ejemplo mantiene separadas las partes principales del proceso:

1. definir el observador;
2. construir la escena celeste;
3. elegir una proyección;
4. dibujar la carta celeste.

Esto facilita la comprensión de cómo Wenu ensambla una carta celeste y de cómo
cada componente puede personalizarse posteriormente.

## Importaciones

Skyfield proporciona la escala de tiempo y la efeméride planetaria utilizadas para definir el
observador. Matplotlib proporciona el lienzo de dibujo.

Wenu proporciona el observador, la esfera celeste, las cuadrículas de coordenadas, los recursos
de datos astronómicos y la proyección estereográfica.

In [ ]:
from pathlib import Path
import traceback

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from wenu.observer import Observer
from wenu.projection import StereographicProjection
from wenu.sky import CelestialSphere
from wenu.sky.coordinate_grids import EclipticGrid, GalacticGrid

from wenu import Viewport
from wenu.renderers.matplotlib_axes import apply_viewport

## Definir la carta celeste

El primer paso es definir el observador y la apariencia de la carta celeste.

Los parámetros siguientes especifican la ubicación de observación, la fecha y la hora, la magnitud
estelar límite, la proyección estereográfica y algunas propiedades visuales de
la carta celeste.

En cuadernos posteriores se analizará cómo afecta cada uno de estos parámetros al resultado
final.

In [ ]:

# ---------------------------------------------------------------------
# Chart
# ---------------------------------------------------------------------

MAGNITUDE_LIMIT = 5.5

PROJECTION_RADIUS = 2.0
FLIP_EAST_WEST = True

SKY_COLOR = "slateblue"

SELECTED_CONSTELLATIONS = None

RIGHT_ASCENSIONS = range(0, 360, 30)
DECLINATIONS = (-60, -30, 0, 30, 60)

## Crear el observador

Una carta astronómica depende tanto de la ubicación de observación como de la hora.

Wenu almacena esta información en un objeto `Observer`. El observador también
proporciona las transformaciones de coordenadas necesarias para convertir las posiciones
celestes en altura y acimut.

In [ ]:
observer = Observer(
    location="La Ligua",
    time="2026-08-15 21:00",
)

In [ ]:
print("UTC time:", observer.t.utc_iso())
print("Latitude:", observer.lat_deg)
print("Longitude:", observer.lon_deg)
print("Elevation:", observer.elevation_m, "m")

## Crear la esfera celeste

`CelestialSphere` es el objeto central de Wenu.

Almacena los objetos astronómicos y las estructuras de referencia que aparecerán
en la carta celeste. A medida que se añaden nuevas capas, pasan a formar parte de la escena
que posteriormente se proyectará y dibujará.

En este cuaderno añadiremos:

- estrellas;
- líneas de constelación;
- límites de constelación;
- puntos de referencia celestes;
- la eclíptica;
- el plano galáctico.

In [ ]:
sky = CelestialSphere(observer)

## Añadir las estrellas

Las estrellas suelen ser la primera capa astronómica que se añade a la esfera celeste.

Este ejemplo utiliza el catálogo Hipparcos y muestra todas las estrellas más brillantes que
la magnitud 5.5.

In [ ]:
stars = sky.add_stars(
    catalog="hipparcos",
    magnitude_limit=MAGNITUDE_LIMIT,
)

El objeto `Stars` carga el catálogo seleccionado, calcula las posiciones aparentes
de las estrellas para el observador y las prepara para su representación gráfica.

Se pueden añadir otros catálogos a Wenu exactamente de la misma manera.

## Añadir las constelaciones

Las figuras de las constelaciones conectan estrellas seleccionadas en patrones reconocibles.

Wenu mantiene las figuras de las constelaciones separadas del catálogo estelar. Las
figuras hacen referencia a identificadores del catálogo, por lo que las estrellas deben añadirse
antes que la capa de constelaciones.

Este ejemplo utiliza el sistema de constelaciones occidentales.

In [ ]:
constellations = sky.add_constellations(
    system="western",
    selected=SELECTED_CONSTELLATIONS,
)

Cuando `SELECTED_CONSTELLATIONS` es `None`, Wenu incluye todas las
constelaciones disponibles en el sistema seleccionado.

En cuadernos posteriores se mostrará cómo dibujar únicamente un grupo de constelaciones elegido.

## Añadir los límites de constelación

Los límites oficiales de la IAU dividen la esfera celeste en 88 regiones de constelaciones.

Estos límites son independientes de las figuras de las constelaciones. Una figura de
constelación es un dibujo cultural, mientras que un límite define una región oficial del
cielo.

In [ ]:
boundaries = sky.add_constellation_boundaries(
    boundaries="iau",
    constellations=SELECTED_CONSTELLATIONS,
)

boundaries.sample()

## Añadir puntos de referencia celestes

Además de estrellas y constelaciones, las cartas astronómicas suelen incluir puntos de referencia
que ayudan a orientar al observador.

Wenu proporciona una colección de puntos de referencia celestes de uso frecuente,
como los polos celestes, los polos de la eclíptica, el centro galáctico y los
puntos cardinales de la eclíptica.

In [ ]:
points = sky.add_points()

El polo celeste visible depende de la latitud del observador. Como este
ejemplo corresponde al hemisferio sur, el polo sur celeste se representará
automáticamente.

In [ ]:
points.add_equatorial_pole(
    pole="visible",
    marker="+",
    label="SCP",
    size=120,
    color="white",
)

A continuación añadimos algunos puntos de referencia adicionales que se muestran con frecuencia en
las cartas astronómicas.

In [ ]:
points.add_ecliptic_pole(
    pole="south",
    marker="+",
    label="SEP",
    size=80,
    color="yellow",
)

points.add_galactic_center(
    marker="+",
    label="GC",
    size=80,
    color="lightblue",
)

points.add_ecliptic_keypoints(
    marker="+",
    size=70,
    color="cyan",
)

## Añadir la eclíptica

La eclíptica es la trayectoria anual aparente del Sol a través de la esfera celeste.

Wenu la construye en coordenadas eclípticas y luego la transforma al cielo aparente del
observador.

In [ ]:
ecliptic_grid = EclipticGrid(
    observer=observer,
    equinox="of_date",
)

ecliptic = ecliptic_grid.ecliptic()

El objeto `EclipticGrid` también puede utilizarse para construir meridianos eclípticos y
círculos de latitud. Aquí usamos únicamente su curva principal: latitud eclíptica
cero.

## Añadir el plano galáctico

El plano galáctico sigue el plano central de la Vía Láctea.

Se define mediante una latitud galáctica igual a cero y se transforma al cielo aparente del
observador del mismo modo que la eclíptica.

In [ ]:
galactic_grid = GalacticGrid(
    observer=observer,
)

galactic_plane = galactic_grid.galactic_plane()

La escena astronómica ya está completa.

Contiene estrellas, figuras de constelaciones, límites oficiales, puntos de referencia
celestes, la eclíptica y el plano galáctico.

El siguiente paso es elegir cómo se proyectará esta esfera celeste sobre
el plano de la carta celeste.

## Elegir una proyección

La esfera celeste es una superficie tridimensional, mientras que una carta celeste es un
dibujo bidimensional.

Una proyección define cómo se trazan las posiciones de la esfera celeste sobre el
plano de la carta celeste.

En este ejemplo utilizamos una proyección estereográfica centrada en el cenit. Esta
proyección conserva los ángulos y representa cada círculo máximo como un círculo o una
línea recta, por lo que resulta especialmente adecuada para las cartas astronómicas.

In [ ]:
projection = StereographicProjection(
    radius=PROJECTION_RADIUS,
    flip_ew=FLIP_EAST_WEST,
)

## Crear el lienzo de dibujo

Wenu se encarga de los cálculos astronómicos y del renderizado de la escena celeste. En este
cuaderno utilizamos Matplotlib para proporcionar el lienzo de dibujo en el que se mostrará la
carta celeste.

El cielo visible se representa mediante un horizonte circular.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

fig.patch.set_alpha(0)

#ax.set_aspect("equal")
ax.axis("off")

#ax.set_xlim(
#    -1.05 * PROJECTION_RADIUS,
#     1.05 * PROJECTION_RADIUS,
#)

#ax.set_ylim(
#    -1.05 * PROJECTION_RADIUS,
#     1.05 * PROJECTION_RADIUS,
#)
R = 1.05 * PROJECTION_RADIUS
viewport = Viewport.centered(
    width=2.0 * R,
    height=2.0 * R,
)

apply_viewport(
    ax,
    viewport,
)

horizon = Circle(
    (0, 0),
    PROJECTION_RADIUS,
    facecolor=SKY_COLOR,
    edgecolor="black",
    linewidth=1.5,
)

_ = ax.add_patch(horizon)
plt.close(fig) # Let us do not draw anything yet...

## Dibujar la esfera celeste

Todo está listo.

La esfera celeste contiene los objetos astronómicos, la proyección los traza sobre
el plano y Matplotlib proporciona la superficie de dibujo.

Renderizar la carta celeste consiste simplemente en pedir a cada capa que se dibuje utilizando
la proyección elegida.

In [ ]:
# Stars, constellation lines, labels, and boundaries
sky.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
import traceback

try:
    sky.draw_equatorial_grid(
        ax=ax,
        projection=projection,
        ra=RIGHT_ASCENSIONS,
        dec=DECLINATIONS,
        color="white",
        linewidth=0.4,
        alpha=0.35,
    )
except Exception:
    traceback.print_exc()

In [ ]:
ecliptic.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="orange",
    linewidth=1.2,
)

In [ ]:
galactic_plane.draw(
    ax=ax,
    projection=projection,
    min_altitude=0.0,
    color="lightblue",
    linewidth=1.2,
)

In [ ]:
points.draw(
    ax=ax,
    projection=projection,
)

In [ ]:
ax.set_title(
    "Southern Sky\n15 August 2026, 21:00 Chile",
    fontsize=16,
)

display(fig)

## Guardar la carta celeste

La carta celeste terminada puede exportarse como una imagen ráster de alta resolución o como un
gráfico vectorial para su publicación.

In [ ]:
output_file = Path("first_chart.png")

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05,
)

print(f"Saved to: {output_file.resolve()}")